# Step 6 Imaging-only DDT From FPD

Read the imaging-only posterior with precomputed FPD values, build a joint 1D grid posterior for a single `D_dt` per sample, and inspect the 6 chains.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

WFI2033_DIR = Path('/mnt/d/lensing/Herculens/Herculensedquasar/WFI2033')
if str(WFI2033_DIR) not in sys.path:
    sys.path.insert(0, str(WFI2033_DIR))

from Tian_infra import import_function
import_function(globals())

jax.config.update('jax_enable_x64', True)
numpyro.enable_x64()

INPUT_PATH = Path('/mnt/lustre/tianli/quasar_hmc/WFI2033_ss=2_inferh0_step6_imaging_only_20260413_19/WFI2033_all_ss=2_inferh0_step6_imaging_only_withFPD.nc')
INPUT_PATH

In [ ]:
Z_LENS = 0.6575
Z_SOURCE = 1.662
OMEGA_M_FIXED = 0.3
H0_LOW = 60.0
H0_HIGH = 80.0
GRID_SIZE = 512

ARCSEC_TO_RAD = np.deg2rad(1.0 / 3600.0)
MPC_TO_KM = 3.0856775814913673e19
DAY_TO_S = 86400.0
TIME_DELAY_SCALE_DAYS = MPC_TO_KM * ARCSEC_TO_RAD**2 / (Cosmo.c_km_s * DAY_TO_S)

TIME_DELAY_OBS = {
    'dt_31_days': {'mean': -36.2, 'sigma_minus': 2.3, 'sigma_plus': 1.6},
    'dt_32_days': {'mean': -37.3, 'sigma_minus': 3.0, 'sigma_plus': 2.6},
    'dt_34_days': {'mean': -59.4, 'sigma_minus': 1.3, 'sigma_plus': 1.3},
}

def fixed_lcdm(h0):
    return {
        'Omegam': jnp.asarray(OMEGA_M_FIXED, dtype=jnp.float64),
        'Omegak': jnp.asarray(0.0, dtype=jnp.float64),
        'w0': jnp.asarray(-1.0, dtype=jnp.float64),
        'wa': jnp.asarray(0.0, dtype=jnp.float64),
        'h0': jnp.asarray(h0, dtype=jnp.float64),
    }

d_dt_low = float(Cosmo.compute_time_delay_distances(fixed_lcdm(H0_HIGH), Z_LENS, Z_SOURCE))
d_dt_high = float(Cosmo.compute_time_delay_distances(fixed_lcdm(H0_LOW), Z_LENS, Z_SOURCE))
d_dt_grid = np.linspace(min(d_dt_low, d_dt_high), max(d_dt_low, d_dt_high), GRID_SIZE)

def loglike_d_dt(d_dt_mpc, fpd_31, fpd_32, fpd_34):
    prefactor_days = d_dt_mpc * TIME_DELAY_SCALE_DAYS
    dt_31 = prefactor_days * fpd_31
    dt_32 = prefactor_days * fpd_32
    dt_34 = prefactor_days * fpd_34
    return (
        Numpyro_function.split_normal_logpdf(dt_31, TIME_DELAY_OBS['dt_31_days']['mean'], TIME_DELAY_OBS['dt_31_days']['sigma_minus'], TIME_DELAY_OBS['dt_31_days']['sigma_plus'])
        + Numpyro_function.split_normal_logpdf(dt_32, TIME_DELAY_OBS['dt_32_days']['mean'], TIME_DELAY_OBS['dt_32_days']['sigma_minus'], TIME_DELAY_OBS['dt_32_days']['sigma_plus'])
        + Numpyro_function.split_normal_logpdf(dt_34, TIME_DELAY_OBS['dt_34_days']['mean'], TIME_DELAY_OBS['dt_34_days']['sigma_minus'], TIME_DELAY_OBS['dt_34_days']['sigma_plus'])
    )

def weighted_quantile(x, w, q):
    order = np.argsort(x)
    x_sorted = np.asarray(x)[order]
    w_sorted = np.asarray(w)[order]
    cdf = np.cumsum(w_sorted)
    cdf = cdf / cdf[-1]
    return np.interp(q, cdf, x_sorted)

def d_dt_posterior_from_fpd(fpd_31, fpd_32, fpd_34):
    logp = np.asarray(loglike_d_dt(jnp.asarray(d_dt_grid), fpd_31, fpd_32, fpd_34), dtype=float)
    logp = logp - np.max(logp)
    weights = np.exp(logp)
    weights = weights / np.sum(weights)
    d_dt_map = float(d_dt_grid[np.argmax(weights)])
    d_dt_mean = float(np.sum(weights * d_dt_grid))
    d_dt_median = float(weighted_quantile(d_dt_grid, weights, 0.5))
    d_dt_lo = float(weighted_quantile(d_dt_grid, weights, 0.16))
    d_dt_hi = float(weighted_quantile(d_dt_grid, weights, 0.84))
    return weights, d_dt_map, d_dt_mean, d_dt_median, d_dt_lo, d_dt_hi

d_dt_low, d_dt_high

In [ ]:
idata = xr.open_dataset(INPUT_PATH)
posterior = idata
n_chain = posterior.sizes['chain']
n_draw = posterior.sizes['draw']

d_dt_weights = np.empty((n_chain, n_draw, GRID_SIZE), dtype=float)
d_dt_map = np.empty((n_chain, n_draw), dtype=float)
d_dt_mean = np.empty((n_chain, n_draw), dtype=float)
d_dt_median = np.empty((n_chain, n_draw), dtype=float)
d_dt_lo = np.empty((n_chain, n_draw), dtype=float)
d_dt_hi = np.empty((n_chain, n_draw), dtype=float)

for c in range(n_chain):
    for d in range(n_draw):
        w, m_map, m_mean, m_med, m_lo, m_hi = d_dt_posterior_from_fpd(
            float(posterior['fpd_31'].isel(chain=c, draw=d).values),
            float(posterior['fpd_32'].isel(chain=c, draw=d).values),
            float(posterior['fpd_34'].isel(chain=c, draw=d).values),
        )
        d_dt_weights[c, d, :] = w
        d_dt_map[c, d] = m_map
        d_dt_mean[c, d] = m_mean
        d_dt_median[c, d] = m_med
        d_dt_lo[c, d] = m_lo
        d_dt_hi[c, d] = m_hi

posterior_ddt = xr.Dataset(
    data_vars={
        'D_dt_Mpc_posterior_weight': (('chain', 'draw', 'd_dt_grid'), d_dt_weights),
        'D_dt_Mpc_map': (('chain', 'draw'), d_dt_map),
        'D_dt_Mpc_mean': (('chain', 'draw'), d_dt_mean),
        'D_dt_Mpc_median': (('chain', 'draw'), d_dt_median),
        'D_dt_Mpc_lo': (('chain', 'draw'), d_dt_lo),
        'D_dt_Mpc_hi': (('chain', 'draw'), d_dt_hi),
    },
    coords={
        'chain': posterior.coords['chain'].values,
        'draw': posterior.coords['draw'].values,
        'd_dt_grid': d_dt_grid,
    },
)
posterior_ddt

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for c in range(n_chain):
    axes[0].plot(posterior_ddt['draw'], posterior_ddt['D_dt_Mpc_map'].isel(chain=c), alpha=0.8, label=f'chain {c}')
    axes[1].plot(posterior_ddt['draw'], posterior_ddt['D_dt_Mpc_mean'].isel(chain=c), alpha=0.8)
    axes[2].plot(posterior_ddt['draw'], posterior_ddt['D_dt_Mpc_median'].isel(chain=c), alpha=0.8)

axes[0].set_ylabel('Ddt MAP [Mpc]')
axes[1].set_ylabel('Ddt mean [Mpc]')
axes[2].set_ylabel('Ddt median [Mpc]')
axes[2].set_xlabel('draw')
axes[0].legend(ncol=3, fontsize=9)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for c in range(n_chain):
    ax.hist(posterior_ddt['D_dt_Mpc_median'].isel(chain=c).values, bins=40, density=True, histtype='step', linewidth=1.8, label=f'chain {c}')
ax.set_xlabel('Ddt median [Mpc]')
ax.set_ylabel('density')
ax.grid(alpha=0.3)
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()